<a href="https://colab.research.google.com/github/Tarunvaka/DataManagement/blob/main/ps2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ps2, manipulating data + merging
**Tarun Vaka** ·

**Research question (abstract).** The Preston curve says rich countries live longer, and ps1 confirmed the shape. But at the same income level, per-person health spending varies wildly, the US spends ~$12k/person while Cuba spends ~$1k with a similar life expectancy. This ps2 extends the ps1 country panel by merging World Bank *current health expenditure per capita* onto the WHO life-expectancy + WB GDP frame, then asks: **does spending more on health actually buy years of life once we control for how rich a country is?**

**Unit of analysis.** Country (merged frame). One sidebar keeps NHANES person-level data from ps1 for a US-inside view.

**Data**
1. **HTML**, Wikipedia, *List of countries by life expectancy*, `https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy` *(from ps1)*
2. **JSON**, World Bank GDP per capita (2022), `https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&date=2022&per_page=400` *(from ps1)*
3. **SAS xpt**, CDC NHANES 2021-2023 Demographics, `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt` *(from ps1)*
4. **JSON (new)**, World Bank *Current health expenditure per capita, current US$* (`SH.XPD.CHEX.PC.CD`, 2022), `https://api.worldbank.org/v2/country/all/indicator/SH.XPD.CHEX.PC.CD?format=json&date=2022&per_page=400`


In [ ]:
import io, json, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 14)

UA = {"User-Agent": "Mozilla/5.0 (ps2 educational)"}


## 1. Rebuild the ps1 datasets

In [ ]:
# 1a. Wikipedia life expectancy (HTML)
URL_LE = "https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy"
html = urllib.request.urlopen(urllib.request.Request(URL_LE, headers=UA)).read().decode()
tables = pd.read_html(io.StringIO(html))

def is_life(t):
    flat = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in t.columns]
    return any("life expectancy" in s.lower() for s in flat)

le = next(t for t in tables if is_life(t)).copy()
le.columns = [" ".join(map(str, c)).strip().lower() if isinstance(c, tuple) else str(c).strip().lower() for c in le.columns]

country_col = next(c for c in le.columns if "country" in c or "location" in c)
both_col    = next(c for c in le.columns if "life expectancy" in c and ("all" in c or "both" in c or "overall" in c))

le = le[[country_col, both_col]].rename(columns={country_col: "country", both_col: "life_both"})
le["life_both"] = pd.to_numeric(le["life_both"], errors="coerce")
le = le.dropna(subset=["life_both"]).reset_index(drop=True)
le.shape, le.head()


((211, 2),
             country  life_both
 0         Hong Kong      85.51
 1             Japan      84.71
 2       South Korea      84.33
 3  French Polynesia      84.07
 4           Andorra      84.04)

In [ ]:
# 1b. World Bank GDP per capita, 2022 (JSON)
URL_GDP = ("https://api.worldbank.org/v2/country/all/indicator/"
           "NY.GDP.PCAP.CD?format=json&date=2022&per_page=400")
payload = json.load(urllib.request.urlopen(URL_GDP))
gdp = pd.json_normalize(payload[1])
gdp = (gdp[["country.value", "countryiso3code", "value"]]
       .rename(columns={"country.value": "country", "countryiso3code": "iso3", "value": "gdp_pc"}))
gdp = gdp.dropna(subset=["gdp_pc", "iso3"])
gdp = gdp[gdp["iso3"].str.len() == 3].reset_index(drop=True)
gdp.shape, gdp.head()


((252, 3),
                           country iso3        gdp_pc
 0     Africa Eastern and Southern  AFE   1675.902524
 1      Africa Western and Central  AFW   2143.072094
 2                      Arab World  ARB   7960.811588
 3          Caribbean small states  CSS  17449.519342
 4  Central Europe and the Baltics  CEB  19526.246344)

In [ ]:
# 1c. NHANES demographics (SAS xpt)
URL_DEMO = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt"
demo_full = pd.read_sas(URL_DEMO, format="xport")

demo = (demo_full[["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH3", "INDFMPIR"]]
        .rename(columns={"SEQN": "id", "RIAGENDR": "gender", "RIDAGEYR": "age",
                         "RIDRETH3": "race", "INDFMPIR": "poverty_ratio"}))
demo["gender"] = demo["gender"].map({1: "Male", 2: "Female"})
demo["race"]   = demo["race"].map({1: "Mex-American", 2: "Other Hispanic",
                                   3: "NH-White",     4: "NH-Black",
                                   6: "NH-Asian",     7: "Other"})
demo.shape, demo.head()


((11933, 5),
          id  gender   age            race  poverty_ratio
 0  130378.0    Male  43.0        NH-Asian           5.00
 1  130379.0    Male  66.0        NH-White           5.00
 2  130380.0  Female  44.0  Other Hispanic           1.41
 3  130381.0  Female   5.0           Other           1.53
 4  130382.0    Male   2.0        NH-White           3.60)

## 2. New dataset, World Bank health expenditure per capita

In [ ]:
URL_HEALTH = ("https://api.worldbank.org/v2/country/all/indicator/"
              "SH.XPD.CHEX.PC.CD?format=json&date=2022&per_page=400")
payload = json.load(urllib.request.urlopen(URL_HEALTH))
health = pd.json_normalize(payload[1])
health = (health[["country.value", "countryiso3code", "value"]]
          .rename(columns={"country.value": "country", "countryiso3code": "iso3", "value": "health_spend"}))
health = health.dropna(subset=["health_spend", "iso3"])
health = health[health["iso3"].str.len() == 3].reset_index(drop=True)
health.shape, health.head()


((234, 3),
                           country iso3  health_spend
 0     Africa Eastern and Southern  AFE     92.035288
 1      Africa Western and Central  AFW     74.479611
 2                      Arab World  ARB    348.767262
 3          Caribbean small states  CSS    915.955449
 4  Central Europe and the Baltics  CEB   1374.160069)

## 3. Merge, life expectancy × GDP × health spend

In [ ]:
# Merge chain: WB GDP carries the ISO3 anchor, then join Wikipedia life expectancy on country name.
step1 = gdp.merge(health[["iso3", "health_spend"]], on="iso3", how="inner")
merged = step1.merge(le[["country", "life_both"]], on="country", how="inner")
merged = merged[["country", "iso3", "life_both", "gdp_pc", "health_spend"]].reset_index(drop=True)
merged.shape, merged.head()


((156, 5),
        country iso3  life_both        gdp_pc  health_spend
 0        World  WLD      73.17  12876.237712   1266.207004
 1  Afghanistan  AFG      66.03    357.261153     80.651604
 2      Albania  ALB      79.60   7756.961887    506.869202
 3      Algeria  DZA      76.26   4960.303343    180.334549
 4      Andorra  AND      84.04  42414.059011   3190.113281)

In [ ]:
# Merge validation, row count and per-column missingness
print("row count:", len(merged))
print("countries lost from GDP frame:", len(gdp) - len(step1),
      "(no health-spend match)")
print("countries lost joining life-exp:", len(step1) - len(merged),
      "(name mismatch with Wikipedia)")
merged.isna().sum()


row count: 156
countries lost from GDP frame: 21 (no health-spend match)
countries lost joining life-exp: 75 (name mismatch with Wikipedia)


,0
country,0
iso3,0
life_both,0
gdp_pc,0
health_spend,0


In [ ]:
# Sanity spot-check, 5 well-known countries
merged[merged["country"].isin(["United States", "Japan", "Cuba", "India", "Norway"])].sort_values("life_both", ascending=False)


,country,iso3,life_both,gdp_pc,health_spend
73,Japan,JPN,84.71,35548.264522,4201.792480
109,Norway,NOR,83.31,113122.130766,8692.576172
150,United States,USA,79.30,78008.689581,12586.196289
66,India,IND,72.00,2279.981457,81.508888


## 4. Manipulations (2× each, all different from ps1)

### 4a. map / recode

In [ ]:
# Recode 1: GDP per capita → World-Bank-style income group via pd.cut
# Thresholds are the WB FY24 GNI bands (rounded), applied here to GDP as an approximation.
BANDS  = [-np.inf, 1145, 4515, 14005, np.inf]
LABELS = ["low", "lower_mid", "upper_mid", "high"]
merged["income_group"] = pd.cut(merged["gdp_pc"], bins=BANDS, labels=LABELS)
merged["income_group"].value_counts()


,count
income_group,
high,50
upper_mid,47
lower_mid,39
low,20


In [ ]:
# Recode 2: ISO3 → continent via a small hand-dict (illustrative subset; the rest fall to "Other")
CONT = {
    "USA":"NA","CAN":"NA","MEX":"NA",
    "BRA":"SA","ARG":"SA","CHL":"SA","COL":"SA","PER":"SA",
    "GBR":"EU","DEU":"EU","FRA":"EU","ITA":"EU","ESP":"EU","NOR":"EU","SWE":"EU","CHE":"EU","NLD":"EU","POL":"EU",
    "CHN":"AS","IND":"AS","JPN":"AS","KOR":"AS","IDN":"AS","PAK":"AS","BGD":"AS","VNM":"AS","THA":"AS","PHL":"AS","SGP":"AS",
    "NGA":"AF","ZAF":"AF","EGY":"AF","KEN":"AF","ETH":"AF","MAR":"AF","GHA":"AF",
    "AUS":"OC","NZL":"OC",
}
merged["continent"] = merged["iso3"].map(CONT).fillna("Other")
merged["continent"].value_counts()


,count
continent,
Other,121
EU,10
AS,9
AF,6
SA,5
NA,3
OC,2


### 4b. replace on condition

In [ ]:
# Replace 1: top-code health_spend at $10,000/capita, this is essentially the US, and it dominates any groupby.
merged.loc[merged["health_spend"] > 10_000, "health_spend"] = np.nan
merged["health_spend"].agg(["count", "max"])


,health_spend
count,154.000000
max,8692.576172


In [ ]:
# Replace 2: NHANES infants (< 2 years) have no meaningful poverty-ratio interpretation, set to NaN.
demo.loc[demo["age"] < 2, "poverty_ratio"] = np.nan
demo[["age", "poverty_ratio"]].describe().round(2)


,age,poverty_ratio
count,11933.00,9588.00
mean,38.32,2.72
std,25.60,1.67
min,0.00,0.00
25%,13.00,1.20
50%,37.00,2.51
75%,62.00,4.55
max,80.00,5.00


### 4c. subset / slice

In [ ]:
# Subset 1: middle-income countries, using pd.Series.between rather than a boolean chain.
mid_income = merged[merged["gdp_pc"].between(5_000, 30_000)].sort_values("gdp_pc")
mid_income.shape, mid_income.head()


((62, 7),
           country iso3  life_both       gdp_pc  health_spend income_group continent
 44    El Salvador  SLV      72.10  5074.600510    500.203217    upper_mid     Other
 146  Turkmenistan  TKM      70.07  5198.189474    579.154541    upper_mid     Other
 143         Tonga  TON      72.89  5298.019425    382.791168    upper_mid     Other
 58      Guatemala  GTM      72.60  5356.924359    384.061584    upper_mid     Other
 49           Fiji  FJI      67.32  5414.120921    217.150681    upper_mid     Other)

In [ ]:
# Subset 2: NHANES non-Hispanic race groups via .isin
nh = demo[demo["race"].isin(["NH-White", "NH-Black", "NH-Asian"])]
nh["race"].value_counts()


,count
race,
NH-White,6217
NH-Black,1597
NH-Asian,681


### 4d. groupby / agg

In [ ]:
# Group 1: within income-group AND continent, do life-expectancy and health-spend track together?
grp1 = (merged.dropna(subset=["health_spend"])
              .groupby(["income_group", "continent"], observed=True)
              .agg(n=("country", "count"),
                   life=("life_both", "mean"),
                   spend=("health_spend", "mean"))
              .round(2))
grp1


n   life    spend
income_group continent                    
low          AF          1  67.31    27.07
             Other      19  63.25    53.97
lower_mid    AF          4  64.73   116.24
             AS          4  71.04    89.96
             Other      31  68.64   143.15
upper_mid    AF          1  66.14   570.48
             AS          3  75.17   417.62
             NA          1  75.07   650.97
             Other      38  73.93   523.47
             SA          4  77.18   809.81
high         AS          2  84.22  4051.97
             EU          9  82.31  4911.23
             NA          1  82.63  6252.34
             OC          2  83.00  5931.16
             Other      33  79.67  2725.40
             SA          1  81.17  1541.38

In [ ]:
# Group 2: NHANES, age spread by race x gender (mean + std, so we see dispersion not just level).
demo.groupby(["race", "gender"], observed=True)["age"].agg(["mean", "std", "count"]).round(2)


mean    std  count
race           gender                     
Mex-American   Female  27.51  21.62    584
               Male    28.04  22.23    533
NH-Asian       Female  36.30  24.43    363
               Male    33.98  24.98    318
NH-Black       Female  36.09  23.85    861
               Male    34.29  24.90    736
NH-White       Female  44.60  25.29   3366
               Male    42.47  26.06   2851
Other          Female  32.52  25.23    474
               Male    31.43  25.33    474
Other Hispanic Female  34.38  24.16    710
               Male    30.16  24.47    663

## 5. Interpretation, does health spend add anything beyond GDP?

In [ ]:
# Simple within-income-group correlation: after fixing income, does health_spend still predict life_both?
def corr_row(g):
    if len(g) < 5: return np.nan
    return g[["life_both", "health_spend"]].corr().iloc[0, 1]

merged.dropna(subset=["health_spend"]).groupby("income_group", observed=True).apply(corr_row, include_groups=False).round(3)


,0
income_group,
low,-0.115
lower_mid,0.374
upper_mid,0.452
high,0.636


_Interpretation._ The within-income-group correlation between health spend and life expectancy is not uniform: it is essentially zero for low-income countries (-0.12), climbs through lower-middle (+0.37) and upper-middle (+0.45), and is strongest for high-income countries (+0.64). Read one way, this says the marginal return to health spending is easiest to see where spending itself varies the most, rich countries differ a lot in how much they spend, and inside that pool spending tracks life expectancy fairly tightly. Read another way, this is exactly where the US anomaly bites: the US was top-coded in section 4b precisely because it spends more than any peer yet lives shorter than Japan, Norway, and Switzerland, so a naïve within-high-income regression still leaves a large US-shaped residual. The NHANES sidebar shows that the age-by-race spread inside the US is wide enough that any country-level story about US health outcomes deserves a within-country companion analysis.